# Classic ML Test Majority Vote Metrics

Notebook повторяет train/test pipeline из `classic_ml_train.ipynb`, но без визуализаций и без grid search.

Обучается только лучшая модель из train-ноутбука: `SVM(C=5.0, gamma='auto', kernel='rbf')`.

Метрики считаются только на test split. Один объект для финальной оценки = одна папка `test_N`; итоговая метка папки выбирается majority vote по `.data` файлам внутри нее.

In [8]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

project_root = Path.cwd()
if not (project_root / 'tools').exists() and (project_root.parent / 'tools').exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from tools.csi_parser import Parser
from tools.filters import median_filter
from tools.classic_ml_inference import run_inference

dataset_root = project_root / 'wifi_data_set_fixed'
median_width = 5
random_state = 42

if median_width % 2 == 0:
    raise ValueError('median_width must be odd')

np.random.seed(random_state)

print(f'project_root: {project_root}')
print(f'dataset_root: {dataset_root}')

project_root: /home/LijnxArcher/CSI-activity-detection
dataset_root: /home/LijnxArcher/CSI-activity-detection/wifi_data_set_fixed


## Helpers

Эти функции повторяют preprocessing из `classic_ml_train.ipynb`.

In [9]:
def parse_path_meta(file_path: Path) -> dict:
    parts = file_path.parts
    return {
        'person_id': next((p for p in parts if p.startswith('id_person_')), 'unknown'),
        'label': next((p for p in parts if p.startswith('label_')), 'unknown'),
        'test_id': next((p for p in parts if p.startswith('test_')), 'unknown'),
    }


def get_test_group_dir(file_path: Path) -> Path:
    for parent in file_path.parents:
        if parent.name.startswith('test_'):
            return parent
    raise ValueError(f'Could not find test_* folder in path: {file_path}')


def skewness(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    z = (x - mu) / sigma
    return float(np.mean(z ** 3))


def kurtosis_excess(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float64)
    mu = x.mean()
    sigma = x.std()
    if sigma == 0:
        return 0.0
    z = (x - mu) / sigma
    return float(np.mean(z ** 4) - 3.0)


def spectral_features(signal: np.ndarray) -> tuple[float, float, float, float, float]:
    x = np.asarray(signal, dtype=np.float64)
    n = len(x)
    if n < 2:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    spectrum = np.fft.rfft(x)
    power = np.abs(spectrum) ** 2
    freqs = np.fft.rfftfreq(n, d=1.0)

    power_sum = power.sum()
    if power_sum <= 0:
        return 0.0, 0.0, 0.0, 0.0, 0.0

    centroid = float((freqs * power).sum() / power_sum)
    spread = float(np.sqrt(((freqs - centroid) ** 2 * power).sum() / power_sum))
    p_norm = power / power_sum
    entropy = float(-(p_norm * np.log2(p_norm + 1e-12)).sum())
    dominant_freq = float(freqs[int(np.argmax(power))])
    rolloff = float(freqs[np.searchsorted(np.cumsum(power), 0.85 * power_sum)])

    return centroid, spread, entropy, dominant_freq, rolloff


def build_unit_signal(df: pd.DataFrame, window: int) -> np.ndarray:
    amp_matrix = np.stack(df['amplitude'].to_numpy(), axis=0).astype(np.float64)
    raw_time_signal = amp_matrix.mean(axis=1)
    return median_filter(raw_time_signal, window)


def build_unit_cache(file_list: list[Path]) -> list[dict]:
    cache = []
    for file_path in file_list:
        df = Parser(file_path).parse()
        filtered_signal = build_unit_signal(df, median_width)
        cache.append({
            'file_path': file_path,
            'df': df,
            'filtered_signal': filtered_signal,
        })
    return cache

## Group train/test split

Split делается по папкам `test_N`, чтобы `.data` файлы одного теста не попадали одновременно в train и test.

In [10]:
all_files = sorted(dataset_root.rglob('*.data'))
if not all_files:
    raise FileNotFoundError(f'No .data files found under {dataset_root}')

file_to_group = {fp: get_test_group_dir(fp) for fp in all_files}
all_groups = sorted(set(file_to_group.values()))
group_labels = [parse_path_meta(group_path)['label'] for group_path in all_groups]

train_groups, test_groups = train_test_split(
    all_groups,
    test_size=0.2,
    random_state=random_state,
    stratify=group_labels,
    shuffle=True,
)

train_group_set = set(train_groups)
test_group_set = set(test_groups)

shared_groups = train_group_set & test_group_set
if shared_groups:
    raise RuntimeError(f'Group leakage detected across split: {sorted(shared_groups)}')

train_files = [fp for fp in all_files if file_to_group[fp] in train_group_set]
test_files = [fp for fp in all_files if file_to_group[fp] in test_group_set]

print(f'Total .data files: {len(all_files)}')
print(f'Total test_* groups: {len(all_groups)}')
print(f'Train groups: {len(train_groups)}')
print(f'Test groups: {len(test_groups)}')
print(f'Train files: {len(train_files)}')
print(f'Test files: {len(test_files)}')

Total .data files: 4800
Total test_* groups: 1600
Train groups: 1280
Test groups: 320
Train files: 3840
Test files: 960


## Feature extraction

Global amplitude min/max считаются только на train. Test использует train-derived scaling.

In [11]:
def build_features_df(unit_cache: list[dict], global_min: float, global_max: float) -> pd.DataFrame:
    den = global_max - global_min
    if den <= 0:
        raise ValueError('Global max equals global min; cannot perform min-max scaling')

    rows = []
    for unit in unit_cache:
        file_path = unit['file_path']
        df = unit['df']
        signal = unit['filtered_signal']

        scaled_signal = (signal - global_min) / den
        scaled_signal = np.clip(scaled_signal, 0.0, 1.0)

        mean_val = float(np.mean(scaled_signal))
        std_val = float(np.std(scaled_signal))
        median_val = float(np.median(scaled_signal))
        min_val = float(np.min(scaled_signal))
        max_val = float(np.max(scaled_signal))
        q25 = float(np.percentile(scaled_signal, 25))
        q75 = float(np.percentile(scaled_signal, 75))
        iqr = q75 - q25
        range_val = max_val - min_val
        rms = float(np.sqrt(np.mean(scaled_signal ** 2)))
        energy = float(np.sum(scaled_signal ** 2))
        zcr = float(np.mean(np.abs(np.diff(np.signbit(scaled_signal - mean_val)).astype(np.int8))))

        sk = skewness(scaled_signal)
        kt = kurtosis_excess(scaled_signal)
        sc, ss, se, dom_freq, rolloff = spectral_features(scaled_signal)
        meta = parse_path_meta(file_path)

        rows.append({
            'file_path': str(file_path),
            'group_path': str(get_test_group_dir(file_path)),
            **meta,
            'n_packets': int(len(df)),
            'mean': mean_val,
            'std': std_val,
            'median': median_val,
            'skew': sk,
            'kurtosis': kt,
            'spectral_centroid': sc,
            'spectral_spread': ss,
            'spectral_entropy': se,
            'dominant_freq': dom_freq,
            'spectral_rolloff_85': rolloff,
            'min': min_val,
            'max': max_val,
            'q25': q25,
            'q75': q75,
            'iqr': iqr,
            'range': range_val,
            'rms': rms,
            'energy': energy,
            'zcr': zcr,
        })

    return pd.DataFrame(rows)


train_unit_cache = build_unit_cache(train_files)
test_unit_cache = build_unit_cache(test_files)

global_min = min(float(np.min(unit['filtered_signal'])) for unit in train_unit_cache)
global_max = max(float(np.max(unit['filtered_signal'])) for unit in train_unit_cache)

train_features_df = build_features_df(train_unit_cache, global_min, global_max)
test_features_df = build_features_df(test_unit_cache, global_min, global_max)

print(f'Train features shape: {train_features_df.shape}')
print(f'Test features shape: {test_features_df.shape}')
print(f'Train-only global amplitude min: {global_min:.6f}')
print(f'Train-only global amplitude max: {global_max:.6f}')

Train features shape: (3840, 25)
Test features shape: (960, 25)
Train-only global amplitude min: 0.691347
Train-only global amplitude max: 39.403599


## PCA preprocessing

`StandardScaler` и `PCA` обучаются только на train. Test только трансформируется.

In [12]:
feature_cols = [
    'mean', 'std', 'median', 'skew', 'kurtosis',
    'spectral_centroid', 'spectral_spread', 'spectral_entropy',
    'dominant_freq', 'spectral_rolloff_85',
    'min', 'max', 'q25', 'q75', 'iqr', 'range',
    'rms', 'energy', 'zcr',
]

X_train_raw = train_features_df[feature_cols].to_numpy(dtype=np.float64)
X_test_raw = test_features_df[feature_cols].to_numpy(dtype=np.float64)

feature_min = X_train_raw.min(axis=0)
feature_max = X_train_raw.max(axis=0)

X_train = np.clip(X_train_raw, feature_min, feature_max)
X_test = np.clip(X_test_raw, feature_min, feature_max)

scaler_pca = StandardScaler()
X_train_std = scaler_pca.fit_transform(X_train)
X_test_std = scaler_pca.transform(X_test)

pca = PCA()
X_train_pca = pca.fit_transform(X_train_std)
X_test_pca = pca.transform(X_test_std)

cum_evr = np.cumsum(pca.explained_variance_ratio_)
k_95 = int(np.searchsorted(cum_evr, 0.95) + 1)

X_train_pca_95 = X_train_pca[:, :k_95]
X_test_pca_95 = X_test_pca[:, :k_95]

y_train = (train_features_df['label'] != 'label_00').astype(int).to_numpy()
y_test_file = (test_features_df['label'] != 'label_00').astype(int).to_numpy()

print(f'k_95: {k_95}')
print(f'X_train_pca_95: {X_train_pca_95.shape}')
print(f'X_test_pca_95: {X_test_pca_95.shape}')
print(f'Train class distribution: 0 -> {(y_train == 0).sum()}, 1 -> {(y_train == 1).sum()}')
print(f'Test file class distribution: 0 -> {(y_test_file == 0).sum()}, 1 -> {(y_test_file == 1).sum()}')

k_95: 6
X_train_pca_95: (3840, 6)
X_test_pca_95: (960, 6)
Train class distribution: 0 -> 960, 1 -> 2880
Test file class distribution: 0 -> 240, 1 -> 720


## Train only the best model

По результатам `classic_ml_train.ipynb` лучшая модель: `SVM(C=5.0, gamma='auto', kernel='rbf')`.

In [13]:
model = SVC(C=5.0, gamma='auto', kernel='rbf', probability=True, random_state=random_state)
model.fit(X_train_pca_95, y_train)

file_pred = model.predict(X_test_pca_95)
print('File-level test metrics before majority vote:')
print(f'accuracy: {accuracy_score(y_test_file, file_pred):.4f}')
print(classification_report(
    y_test_file,
    file_pred,
    labels=[0, 1],
    target_names=['static / no motion', 'dynamic / motion'],
    zero_division=0,
))

File-level test metrics before majority vote:
accuracy: 0.9635
                    precision    recall  f1-score   support

static / no motion       0.94      0.92      0.93       240
  dynamic / motion       0.97      0.98      0.98       720

          accuracy                           0.96       960
         macro avg       0.95      0.95      0.95       960
      weighted avg       0.96      0.96      0.96       960



## Save notebook artifacts for `classic_ml_inference.py`

Для majority vote используем существующий inference-код: `run_inference` из `tools/classic_ml_inference.py`. Поэтому сохраняем модель и preprocessing bundle в отдельную папку артефактов этого notebook.

In [14]:
artifacts_dir = project_root / 'artifacts' / 'classic_ml_majority_vote_metrics'
artifacts_dir.mkdir(parents=True, exist_ok=True)

model_path = artifacts_dir / 'svm_model.joblib'
preprocessing_path = artifacts_dir / 'preprocessing.joblib'
metadata_path = artifacts_dir / 'metadata.json'

joblib.dump(model, model_path)

preproc_bundle = {
    'median_width': median_width,
    'global_min': float(global_min),
    'global_max': float(global_max),
    'feature_cols': feature_cols,
    'feature_min': feature_min,
    'feature_max': feature_max,
    'scaler_pca': scaler_pca,
    'pca': pca,
    'k_95': int(k_95),
    'model_type': 'sklearn',
}
joblib.dump(preproc_bundle, preprocessing_path)

metadata = {
    'source_notebook': 'classic_ml_majority_vote_metrics.ipynb',
    'model': 'SVM',
    'best_params_from_classic_ml_train': {'C': 5.0, 'gamma': 'auto', 'kernel': 'rbf'},
    'split': 'group train/test split by test_N folders, test_size=0.2, random_state=42',
    'model_path': str(model_path),
    'preprocessing_path': str(preprocessing_path),
}
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')

print(f'model_path: {model_path}')
print(f'preprocessing_path: {preprocessing_path}')
print(f'metadata_path: {metadata_path}')

model_path: /home/LijnxArcher/CSI-activity-detection/artifacts/classic_ml_majority_vote_metrics/svm_model.joblib
preprocessing_path: /home/LijnxArcher/CSI-activity-detection/artifacts/classic_ml_majority_vote_metrics/preprocessing.joblib
metadata_path: /home/LijnxArcher/CSI-activity-detection/artifacts/classic_ml_majority_vote_metrics/metadata.json


## Majority vote on test groups only

Каждый `.data` файл из test split прогоняется через `run_inference`. Затем внутри каждой папки `test_N` выбирается итоговая метка по большинству.

In [15]:
def true_class_from_label(label_name: str) -> int:
    return 0 if label_name == 'label_00' else 1


rows = []
test_groups_sorted = sorted(test_group_set)

for idx, test_group in enumerate(test_groups_sorted, start=1):
    data_files = sorted(test_group.glob('*.data'))
    if not data_files:
        continue

    file_results = [run_inference(model_path, preprocessing_path, data_file) for data_file in data_files]
    file_predictions = [int(result['predicted_class']) for result in file_results]
    votes_static = file_predictions.count(0)
    votes_dynamic = file_predictions.count(1)
    y_pred = 1 if votes_dynamic >= votes_static else 0

    meta = parse_path_meta(test_group)
    y_true = true_class_from_label(meta['label'])

    rows.append({
        'person_id': meta['person_id'],
        'label': meta['label'],
        'test_id': meta['test_id'],
        'group_path': str(test_group),
        'n_files': len(data_files),
        'votes_static': votes_static,
        'votes_dynamic': votes_dynamic,
        'y_true': y_true,
        'y_pred': y_pred,
    })

    if idx % 50 == 0:
        print(f'processed test groups: {idx}/{len(test_groups_sorted)}')

vote_results_df = pd.DataFrame(rows)
print(f'Test groups scored: {len(vote_results_df)}')
vote_results_df.head()

processed test groups: 50/320
processed test groups: 100/320
processed test groups: 150/320
processed test groups: 200/320
processed test groups: 250/320
processed test groups: 300/320
Test groups scored: 320


,person_id,label,test_id,group_path,n_files,votes_static,votes_dynamic,y_true,y_pred
0,id_person_01,label_00,test_02,/home/LijnxArcher/CSI-activity-detection/wifi_...,3,2,1,0,0
1,id_person_01,label_00,test_13,/home/LijnxArcher/CSI-activity-detection/wifi_...,3,3,0,0,0
2,id_person_01,label_00,test_20,/home/LijnxArcher/CSI-activity-detection/wifi_...,3,3,0,0,0
3,id_person_01,label_00,test_21,/home/LijnxArcher/CSI-activity-detection/wifi_...,3,3,0,0,0
4,id_person_01,label_00,test_34,/home/LijnxArcher/CSI-activity-detection/wifi_...,3,3,0,0,0


## Test metrics after majority vote

In [16]:
y_test_group = vote_results_df['y_true'].to_numpy()
y_pred_group = vote_results_df['y_pred'].to_numpy()

precision, recall, f1, support = precision_recall_fscore_support(
    y_test_group,
    y_pred_group,
    labels=[0, 1],
    zero_division=0,
)

metrics_df = pd.DataFrame({
    'class': ['static / no motion', 'dynamic / motion'],
    'precision': precision,
    'recall': recall,
    'f1_score': f1,
    'support': support,
})

print('Confusion matrix, rows=true, cols=pred:')
print(confusion_matrix(y_test_group, y_pred_group, labels=[0, 1]))
print()
print(classification_report(
    y_test_group,
    y_pred_group,
    labels=[0, 1],
    target_names=['static / no motion', 'dynamic / motion'],
    zero_division=0,
))

metrics_df

Confusion matrix, rows=true, cols=pred:
[[ 78   2]
 [  0 240]]

                    precision    recall  f1-score   support

static / no motion       1.00      0.97      0.99        80
  dynamic / motion       0.99      1.00      1.00       240

          accuracy                           0.99       320
         macro avg       1.00      0.99      0.99       320
      weighted avg       0.99      0.99      0.99       320



,class,precision,recall,f1_score,support
0,static / no motion,1.000000,0.975,0.987342,80
1,dynamic / motion,0.991736,1.000,0.995851,240


## Test errors

In [17]:
errors_df = vote_results_df[vote_results_df['y_true'] != vote_results_df['y_pred']].copy()
print(f'Test group errors: {len(errors_df)}')
errors_df

Test group errors: 2


,person_id,label,test_id,group_path,n_files,votes_static,votes_dynamic,y_true,y_pred
161,id_person_03,label_00,test_17,/home/LijnxArcher/CSI-activity-detection/wifi_...,3,1,2,0,1
174,id_person_03,label_00,test_76,/home/LijnxArcher/CSI-activity-detection/wifi_...,3,1,2,0,1
